## GPT2示例01-推理下一个概率最大的词

In [1]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning, message="torch.utils._pytree._register_pytree_node is deprecated")
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
import torch

from transformers import GPT2Tokenizer, GPT2LMHeadModel

### 1. 加载GPT2模型并分词

In [3]:
# 加载预训练模型:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

In [4]:
# !curl https://huggingface.co

In [5]:
# 编码一个句子
text = "Python is a"

In [6]:
input_ids = tokenizer.encode(text)
input_ids

[37906, 318, 257]

In [7]:
# 把ids解码为字符
tokenizer.decode(input_ids)

'Python is a'

In [8]:
# 把输入的字符ids转换为张量
input_ids_tensor = torch.tensor(input_ids).unsqueeze(0)

In [9]:
input_ids_tensor

tensor([[37906,   318,   257]])

### 2. 加载模型并执行推理

In [10]:
# 加载预训练的模型
model = GPT2LMHeadModel.from_pretrained("gpt2")

In [11]:
# 让模型执行推理(预测)，而不是进行训练
model.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [12]:
# 使用不会为模型计算梯度和执行反向传播的上下文。
# 关闭梯度计算，这会节省点计算资源，可以使推理运行得更快
with torch.no_grad():
    outputs = model(input_ids_tensor)

In [13]:
type(outputs)

transformers.modeling_outputs.CausalLMOutputWithCrossAttentions

In [14]:
len(outputs)

2

In [15]:
outputs[0].shape

torch.Size([1, 3, 50257])

In [16]:
type(outputs[1])

tuple

In [17]:
type(outputs[1]), len(outputs[1][0])

(tuple, 2)

In [18]:
outputs[1][0][0].shape

torch.Size([1, 12, 3, 64])

In [19]:
# 模型执行完后的预测
predictions = outputs[0]

In [20]:
predictions.shape

torch.Size([1, 3, 50257])

> 模型推理后的预测(`predictions`)是一个张量。其中值的大小是描述每个单词出现的概率。    
> 数组：`[1, 3, 50257]`是`[batch_size, sequence_length, config.vocab_size]`。`batch_size`是1，我们传入的句子是一个，序列长度是3，我们句子的token长度是3.

In [21]:
predictions[0,:,:]

tensor([[ -32.3188,  -32.0519,  -34.2341,  ...,  -39.3484,  -39.0407,
          -32.3083],
        [-118.8541, -119.4894, -120.9461,  ..., -128.0332, -125.2607,
         -121.2128],
        [-123.1338, -122.4541, -123.7767,  ..., -132.1330, -128.7352,
         -123.2540]])

In [22]:
predictions[0, -1, :]

tensor([-123.1338, -122.4541, -123.7767,  ..., -132.1330, -128.7352,
        -123.2540])

In [23]:
# 获取概率最大的那个索引
predictions_max_index = torch.argmax(predictions[0, -1, :]).item()

In [24]:
predictions_max_index

845

In [25]:
# 预测的文本
predicted_text = tokenizer.decode(input_ids + [predictions_max_index])
predicted_text

'Python is a very'

### 3. 连续执行N次推理

In [26]:
# 执行十次试一下
input_ids_list = input_ids.copy()
print(tokenizer.decode(input_ids_list))

for i in range(20):
    input_ids_t = torch.tensor(input_ids_list).unsqueeze(0)
    with torch.no_grad():
        outputs = model(input_ids_t)
        predictions = outputs[0]
        predicted_max_index = torch.argmax(predictions[0, -1, :]).item()
        input_ids_list.append(predicted_max_index)
        # 输出文本
    predicted_text = tokenizer.decode(input_ids_list)
    print(predicted_text)
    

Python is a
Python is a very
Python is a very simple
Python is a very simple and
Python is a very simple and powerful
Python is a very simple and powerful language
Python is a very simple and powerful language.
Python is a very simple and powerful language. It
Python is a very simple and powerful language. It is
Python is a very simple and powerful language. It is a
Python is a very simple and powerful language. It is a very
Python is a very simple and powerful language. It is a very simple
Python is a very simple and powerful language. It is a very simple language
Python is a very simple and powerful language. It is a very simple language that
Python is a very simple and powerful language. It is a very simple language that is
Python is a very simple and powerful language. It is a very simple language that is very
Python is a very simple and powerful language. It is a very simple language that is very easy
Python is a very simple and powerful language. It is a very simple language that

In [27]:
input_ids

[37906, 318, 257]

In [28]:
print(input_ids_list)

[37906, 318, 257, 845, 2829, 290, 3665, 3303, 13, 632, 318, 257, 845, 2829, 3303, 326, 318, 845, 2562, 284, 2193, 290, 779]


### 4. 封装出一个推理N次的测试函数

In [29]:
def predict_text(source, count=10, one_line=False):
    # 分词
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    # 模型
    model = GPT2LMHeadModel.from_pretrained("gpt2")
    model.eval()

    token_ids = tokenizer.encode(source)

    # 执行N次推理
    print(source, end="" if one_line else "\n")
    for i in range(count):
        token_ids_tensor = torch.tensor(token_ids).unsqueeze(0)
        with torch.no_grad():
            outputs = model(token_ids_tensor)
            predictions = outputs[0]
            predicted_max_index = torch.argmax(predictions[0, -1, :]).item()
            token_ids.append(predicted_max_index)
        # 输出文本
        if one_line:
            print(tokenizer.decode([predicted_max_index]), end='')
        else:
            predicted_text = tokenizer.decode(token_ids)
            print(predicted_text)

In [30]:
predict_text("Python is", count=20)

Python is
Python is a
Python is a very
Python is a very simple
Python is a very simple and
Python is a very simple and powerful
Python is a very simple and powerful language
Python is a very simple and powerful language.
Python is a very simple and powerful language. It
Python is a very simple and powerful language. It is
Python is a very simple and powerful language. It is a
Python is a very simple and powerful language. It is a very
Python is a very simple and powerful language. It is a very simple
Python is a very simple and powerful language. It is a very simple language
Python is a very simple and powerful language. It is a very simple language that
Python is a very simple and powerful language. It is a very simple language that is
Python is a very simple and powerful language. It is a very simple language that is very
Python is a very simple and powerful language. It is a very simple language that is very easy
Python is a very simple and powerful language. It is a very simple lan

In [31]:
predict_text("Python is", count=22, one_line=True)

Python is a very simple and powerful language. It is a very simple language that is very easy to learn and use.